# Synthetic Data Generation → Object Detection Training

Fine-tune an object detector on **synthetic data generated with NVIDIA Omniverse Replicator**,
using **TAO Toolkit** (`detectnet_v2`, ResNet-18 backbone).

| Step | What happens | Where it runs |
|---|---|---|
| 1 | Configure paths & hyperparameters | host |
| 2 | Set up TAO Toolkit via a Docker container | host → container |
| 3 | Download a pre-trained object detection model | host (NGC) |
| 4 | Convert the dataset to KITTI, then to **TFRecords** | host + container |
| 5 | Specify training parameters (batch size, learning rate, …) | host |
| 6 | Train the model with TAO Toolkit | container (GPU) |
| 7 | Evaluate on held-out test data | container (GPU) |
| 8 | Visualize results | host |

At the end you will have either a fine-tuned detector of your own, or you can fall back to the
pre-trained model shipped with the module.

> **Runtime.** Training with the default parameters takes roughly **one hour on an NVIDIA RTX A6000**.
> Fewer epochs (`NUM_EPOCHS`) trade accuracy for wall-clock time.

> **Remote / cloud instances.** If you are running this on a cloud or remote machine, the container
> setup in Step 2 is the only part that changes — mount your data volume and forward the Jupyter
> port. See the NVIDIA "Running TAO Toolkit on a cloud instance" guide.

---
## 0. Prerequisites check

Verifies the host has what the rest of the notebook assumes: an NVIDIA GPU, a working Docker with
the container toolkit, and enough disk. **Read the architecture warning if one appears** — it
determines whether Step 2 can work at all.

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

def _sh(cmd):
    """Run a command, return (ok, combined output)."""
    try:
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        return out.returncode == 0, (out.stdout + out.stderr).strip()
    except Exception as exc:  # noqa: BLE001
        return False, str(exc)

ARCH = platform.machine()
print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.system()} {ARCH}")

ok, out = _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
print(f"GPU         : {out.splitlines()[0] if ok and out else 'NOT FOUND'}")

ok_docker, out = _sh("docker --version")
print(f"Docker      : {out if ok_docker else 'NOT FOUND'}")

ok_rt, _ = _sh("docker run --rm --gpus all ubuntu:22.04 nvidia-smi -L")
print(f"GPU in Docker: {'yes' if ok_rt else 'no — install nvidia-container-toolkit'}")

free_gb = shutil.disk_usage(Path.cwd()).free / 1e9
print(f"Free disk   : {free_gb:.0f} GB  {'(ok)' if free_gb > 50 else '(LOW — need ~50 GB)'}")

if ARCH != "x86_64":
    print(
        f"\n!! ARCHITECTURE WARNING: this host is {ARCH}.\n"
        "   The TAO Toolkit TF1 container (detectnet_v2) is published for linux/amd64 ONLY.\n"
        "   Options:\n"
        "     a) Run Steps 2-7 on an x86_64 machine or cloud instance (recommended), then copy\n"
        "        the trained model back here for Step 8.\n"
        "     b) Skip training entirely and use the pre-trained model shipped with the module\n"
        "        (Step 3 + Step 8 still work on this host).\n"
        "   Steps 0, 1, 4a-4c and 8 are pure Python and run fine on any architecture."
    )

---
## 1. Configuration

Everything downstream reads from this cell. Copy `.env.example` to `.env` to override any of it
without editing the notebook.

`NGC_API_KEY` is needed to pull the TAO container and the pre-trained backbone —
generate one at <https://ngc.nvidia.com/setup/api-key>.

In [ ]:
# Load .env if present (optional; plain os.environ works too).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def env(key, default):
    return os.environ.get(key) or default

# --- Paths --------------------------------------------------------------------
PROJECT_DIR = Path(env("LOCAL_PROJECT_DIR", Path.cwd().parent)).resolve()
# Path the project is mounted at *inside* the container. Every path handed to TAO
# must be expressed relative to this, not to the host.
TAO_DIR = env("TAO_PROJECT_DIR", "/workspace/tao-experiments")

RAW_DATA_DIR = PROJECT_DIR / env("RAW_DATA_DIR", "data/raw")        # Replicator output
KITTI_DIR    = PROJECT_DIR / env("KITTI_DATA_DIR", "data/kitti")    # converted, KITTI layout
TFRECORDS_DIR= PROJECT_DIR / "data/tfrecords"
PRETRAINED_DIR = PROJECT_DIR / "pretrained_models"
RESULTS_DIR  = PROJECT_DIR / "results"
SPECS_DIR    = PROJECT_DIR / "specs"

for d in (RAW_DATA_DIR, KITTI_DIR, TFRECORDS_DIR, PRETRAINED_DIR, RESULTS_DIR, SPECS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Dataset ------------------------------------------------------------------
TARGET_CLASS = env("TARGET_CLASS", "palletjack")
VAL_SPLIT    = float(env("VAL_SPLIT", "0.2"))
RANDOM_SEED  = int(env("RANDOM_SEED", "42"))
IMAGE_WIDTH  = 1280   # must be divisible by 16 for detectnet_v2
IMAGE_HEIGHT = 720

# --- Model / training ---------------------------------------------------------
ARCH_LAYERS  = int(env("ARCH", "resnet18").replace("resnet", ""))
BATCH_SIZE   = int(env("BATCH_SIZE", "4"))
NUM_EPOCHS   = int(env("NUM_EPOCHS", "80"))
LEARNING_RATE= float(env("LEARNING_RATE", "5e-4"))
NUM_GPUS     = int(env("NUM_GPUS", "1"))

# TAO's model-encryption key. Any string; you need the SAME one to load the model later.
KEY = env("TAO_KEY", "nvidia_tlt")

# --- Container ----------------------------------------------------------------
TAO_IMAGE = env("TAO_DOCKER_IMAGE", "nvcr.io/nvidia/tao/tao-toolkit:5.0.0-tf1.15.5")
NGC_API_KEY = env("NGC_API_KEY", "")

assert 0.0 < VAL_SPLIT < 1.0, "VAL_SPLIT must be a fraction strictly between 0 and 1"
assert IMAGE_WIDTH % 16 == 0 and IMAGE_HEIGHT % 16 == 0, \
    "detectnet_v2 requires image dimensions divisible by 16"

print(f"Project dir  : {PROJECT_DIR}")
print(f"Mounted at   : {TAO_DIR}  (inside container)")
print(f"Raw data     : {RAW_DATA_DIR}")
print(f"KITTI data   : {KITTI_DIR}")
print(f"Class        : {TARGET_CLASS}")
print(f"Image size   : {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
print(f"Batch/epochs : {BATCH_SIZE} / {NUM_EPOCHS} @ lr={LEARNING_RATE}")
print(f"NGC key      : {'set' if NGC_API_KEY else 'NOT SET — Steps 2 and 3 will fail'}")

---
## 2. Set up TAO Toolkit via a Docker container

TAO ships as a container; we drive it with plain `docker run` rather than the `nvidia-tao`
launcher, which keeps this notebook working even where the launcher wheel is unavailable.

Two things get set up here:

1. **`~/.tao_mounts.json`** — the mount map, so the `tao` CLI works too if you prefer it.
2. **A `tao(...)` helper** — runs a TAO subcommand in the container with the project bind-mounted
   and streams the log back into the notebook.

In [ ]:
# 2.1 — Mount map for the `tao` launcher CLI (harmless if you only use the helper below).
mounts = {
    "Mounts": [
        {"source": str(PROJECT_DIR), "destination": TAO_DIR},
    ],
    "DockerOptions": {
        "shm_size": "16G",
        "ulimits": {"memlock": -1, "stack": 67108864},
        "user": f"{os.getuid()}:{os.getgid()}",
    },
}
mounts_path = Path.home() / ".tao_mounts.json"
mounts_path.write_text(json.dumps(mounts, indent=2))
print(f"wrote {mounts_path}")
print(json.dumps(mounts, indent=2))

In [ ]:
# 2.2 — Authenticate to NGC and pull the TAO container (~10 GB, one time).
if NGC_API_KEY:
    login = subprocess.run(
        "docker login nvcr.io --username '$oauthtoken' --password-stdin",
        input=NGC_API_KEY, shell=True, capture_output=True, text=True,
    )
    print(login.stdout or login.stderr)
else:
    print("NGC_API_KEY not set — skipping docker login (pull will fail if the image is gated)")

In [ ]:
def run(cmd, check=True):
    """Run a shell command, streaming stdout+stderr into the notebook."""
    print(f"$ {cmd}\n", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit {proc.returncode}: {cmd}")
    return proc.returncode

def tao(subcommand, gpus="all"):
    """Run a TAO Toolkit subcommand inside the container.

    Paths inside `subcommand` must be container paths (under TAO_DIR), not host paths.
    """
    docker_cmd = (
        f"docker run --rm --gpus {gpus} "
        f"--shm-size=16g --ulimit memlock=-1 --ulimit stack=67108864 "
        f"-u {os.getuid()}:{os.getgid()} "
        f"-v {PROJECT_DIR}:{TAO_DIR} "
        f"-w {TAO_DIR} "
        f"-e NVIDIA_VISIBLE_DEVICES=all "
        f"{TAO_IMAGE} {subcommand}"
    )
    return run(docker_cmd)

def to_container(path):
    """Translate a host path under PROJECT_DIR into its in-container equivalent."""
    return str(Path(TAO_DIR) / Path(path).resolve().relative_to(PROJECT_DIR))

print("helpers ready: run(), tao(), to_container()")

In [ ]:
# 2.3 — Pull the image and confirm the toolkit answers.
run(f"docker pull {TAO_IMAGE}")
tao("detectnet_v2 --help")

---
## 3. Download a pre-trained object detection model

We start from an ImageNet-pretrained **ResNet-18** backbone rather than random weights — with a
few thousand synthetic images that is the difference between a detector that works and one that
does not.

The download is a public NGC artifact, so plain `wget` is enough; the `ngc` CLI path is shown as a
fallback.

In [ ]:
model_zip = PRETRAINED_DIR / f"pretrained_detectnet_v2_resnet{ARCH_LAYERS}.zip"
model_dir = PRETRAINED_DIR / f"pretrained_detectnet_v2_resnet{ARCH_LAYERS}"

if not model_dir.exists():
    url = (
        "https://api.ngc.nvidia.com/v2/models/nvidia/tao/pretrained_detectnet_v2/"
        f"versions/resnet{ARCH_LAYERS}/zip"
    )
    run(f"wget --content-disposition '{url}' -O {model_zip} -q --show-progress")
    run(f"unzip -o {model_zip} -d {model_dir}")
    model_zip.unlink(missing_ok=True)
else:
    print(f"already present: {model_dir}")

# Locate the .hdf5 weights the spec file will point at.
weights = sorted(model_dir.rglob("*.hdf5"))
assert weights, f"no .hdf5 weights found under {model_dir}"
PRETRAINED_WEIGHTS = weights[0]
print(f"\npretrained backbone: {PRETRAINED_WEIGHTS}")
print(f"  size: {PRETRAINED_WEIGHTS.stat().st_size / 1e6:.1f} MB")
print(f"  container path: {to_container(PRETRAINED_WEIGHTS)}")

<details>
<summary>Alternative: download with the NGC CLI</summary>

```bash
ngc registry model download-version \
    nvidia/tao/pretrained_detectnet_v2:resnet18 \
    --dest pretrained_models/
```
</details>

---
## 4. Convert the dataset to TFRecords

Two hops: **Replicator output → KITTI → TFRecords**. TFRecords is a sharded binary format that
lets the training loop stream data without thousands of small-file reads — it is what makes each
epoch fast.

Expected Replicator (`BasicWriter`) layout under `data/raw/`:

```
data/raw/<any-subdir>/
├── rgb_0000.png
├── bounding_box_2d_tight_0000.npy
├── bounding_box_2d_tight_labels_0000.json
└── ...
```

Target KITTI layout:

```
data/kitti/
├── images/  frame_000000.png
└── labels/  frame_000000.txt   # "palletjack 0 0 0 x1 y1 x2 y2 0 0 0 0 0 0 0"
```

In [ ]:
# 4.1 — Replicator -> KITTI converter.
import numpy as np
from PIL import Image

# KITTI label columns: type truncated occluded alpha x1 y1 x2 y2 h w l x y z ry
KITTI_LINE = "{cls} 0.00 0 0.00 {x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f} 0.00 0.00 0.00 0.00 0.00 0.00 0.00"

def _boxes_from_npy(npy_path):
    """Return [(semantic_id, x1, y1, x2, y2, occlusion), ...] from a Replicator bbox file."""
    arr = np.load(str(npy_path), allow_pickle=True)
    out = []
    names = arr.dtype.names
    for row in arr:
        if names:  # structured array — the normal case
            sid = int(row["semanticId"])
            x1, y1, x2, y2 = (float(row[k]) for k in ("x_min", "y_min", "x_max", "y_max"))
            occ = float(row["occlusionRatio"]) if "occlusionRatio" in names else 0.0
        else:      # plain Nx5/Nx6 array
            sid, x1, y1, x2, y2 = (float(v) for v in row[:5])
            occ = float(row[5]) if len(row) > 5 else 0.0
        out.append((int(sid), x1, y1, x2, y2, occ))
    return out

def convert_replicator_to_kitti(
    raw_dir, out_dir, target_class,
    max_occlusion=0.9, min_box_px=8, image_size=None,
):
    """Convert Replicator BasicWriter output to a flat KITTI dataset.

    Boxes are scaled to `image_size`, clamped to the frame, then dropped if they are
    more than `max_occlusion` occluded, degenerate (zero area after clamping), or
    smaller than `min_box_px` on either side. Degenerate boxes poison detectnet_v2's
    rasterizer rather than being ignored by it.

    `min_box_px` is measured in **output** pixels, i.e. after resizing — the same
    resolution the spec file's `minimum_bounding_box_height` applies to, so keep the
    two coherent. Note the consequence when upscaling: at 640->1280 a 4 px source box
    becomes 8 px and survives, despite carrying no real signal. Set `min_box_px`
    relative to your output resolution, not your source.

    Returns a stats dict.
    """
    raw_dir, out_dir = Path(raw_dir), Path(out_dir)
    img_out, lbl_out = out_dir / "images", out_dir / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    rgb_files = sorted(raw_dir.rglob("rgb_*.png")) + sorted(raw_dir.rglob("rgb_*.jpg"))
    stats = {"frames": 0, "empty_frames": 0, "boxes_kept": 0, "boxes_dropped": 0, "missing": 0}

    for idx, rgb in enumerate(rgb_files):
        suffix = rgb.stem.split("_")[-1]                      # "0000"
        npy = rgb.parent / f"bounding_box_2d_tight_{suffix}.npy"
        js  = rgb.parent / f"bounding_box_2d_tight_labels_{suffix}.json"
        if not npy.exists():
            stats["missing"] += 1
            continue

        # semanticId -> class name
        id2class = {}
        if js.exists():
            for k, v in json.loads(js.read_text()).items():
                id2class[int(k)] = (v.get("class") if isinstance(v, dict) else str(v)).lower()

        with Image.open(rgb) as im:
            W, H = im.size
            stem = f"frame_{idx:06d}"
            dst = img_out / f"{stem}.png"
            if image_size and (W, H) != tuple(image_size):
                im = im.convert("RGB").resize(tuple(image_size), Image.BILINEAR)
                sx, sy = image_size[0] / W, image_size[1] / H
                W, H = image_size
            else:
                im = im.convert("RGB")
                sx = sy = 1.0
            im.save(dst)

        lines = []
        for sid, x1, y1, x2, y2, occ in _boxes_from_npy(npy):
            name = id2class.get(sid, target_class)
            if name != target_class:
                stats["boxes_dropped"] += 1
                continue
            x1, x2 = sorted((x1 * sx, x2 * sx))
            y1, y2 = sorted((y1 * sy, y2 * sy))
            x1, y1 = max(0.0, x1), max(0.0, y1)
            x2, y2 = min(float(W - 1), x2), min(float(H - 1), y2)
            w, h = x2 - x1, y2 - y1
            if occ > max_occlusion or w <= 0 or h <= 0 or w < min_box_px or h < min_box_px:
                stats["boxes_dropped"] += 1
                continue
            lines.append(KITTI_LINE.format(cls=target_class, x1=x1, y1=y1, x2=x2, y2=y2))
            stats["boxes_kept"] += 1

        (lbl_out / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        stats["frames"] += 1
        if not lines:
            stats["empty_frames"] += 1

    return stats

print("converter defined")

In [ ]:
# 4.2 — Run the conversion.
raw_frames = list(RAW_DATA_DIR.rglob("rgb_*.png"))
if not raw_frames:
    print(
        f"No Replicator frames under {RAW_DATA_DIR}.\n"
        "Generate them first with Omniverse Replicator, or drop an existing KITTI dataset\n"
        f"straight into {KITTI_DIR}/{{images,labels}} and skip to 4.3."
    )
else:
    stats = convert_replicator_to_kitti(
        RAW_DATA_DIR, KITTI_DIR, TARGET_CLASS, image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    )
    print(json.dumps(stats, indent=2))
    if stats["frames"] and stats["empty_frames"] / stats["frames"] > 0.5:
        print(
            "\nWARNING: over half the frames have no boxes. Check that TARGET_CLASS matches the"
            " semantic label used in your Replicator scene."
        )

In [ ]:
# 4.3 — Dataset sanity check: counts, box statistics, class balance.
images = sorted((KITTI_DIR / "images").glob("*.png"))
labels = sorted((KITTI_DIR / "labels").glob("*.txt"))
assert images, f"no images in {KITTI_DIR / 'images'}"
assert len(images) == len(labels), f"{len(images)} images vs {len(labels)} labels — mismatch"

box_count, widths, heights, classes = 0, [], [], {}
for lbl in labels:
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if len(parts) < 8:
            continue
        classes[parts[0]] = classes.get(parts[0], 0) + 1
        x1, y1, x2, y2 = (float(v) for v in parts[4:8])
        widths.append(x2 - x1)
        heights.append(y2 - y1)
        box_count += 1

print(f"images       : {len(images)}")
print(f"boxes        : {box_count}  ({box_count / max(len(images), 1):.2f} per image)")
print(f"classes      : {classes}")
if widths:
    print(f"box width  px: min {min(widths):.0f} / median {sorted(widths)[len(widths)//2]:.0f} / max {max(widths):.0f}")
    print(f"box height px: min {min(heights):.0f} / median {sorted(heights)[len(heights)//2]:.0f} / max {max(heights):.0f}")
    tiny = sum(1 for h in heights if h < 20)
    if tiny:
        print(
            f"\nNote: {tiny} boxes are under 20 px tall and will be filtered by the evaluation"
            " config's minimum_height. Lower it if small objects matter for your use case."
        )

In [ ]:
# 4.4 — Eyeball the ground truth before training on it.
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

def show_kitti_samples(kitti_dir, n=6, seed=RANDOM_SEED, title="Ground truth"):
    kitti_dir = Path(kitti_dir)
    imgs = sorted((kitti_dir / "images").glob("*.png"))
    if not imgs:
        print("nothing to show")
        return
    picks = random.Random(seed).sample(imgs, min(n, len(imgs)))
    cols = 3
    rows = (len(picks) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3 * rows))
    for ax, img_path in zip(np.array(axes).ravel(), picks):
        ax.imshow(Image.open(img_path))
        lbl = kitti_dir / "labels" / f"{img_path.stem}.txt"
        n_boxes = 0
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) < 8:
                    continue
                x1, y1, x2, y2 = (float(v) for v in p[4:8])
                ax.add_patch(patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor="lime", facecolor="none",
                ))
                n_boxes += 1
        ax.set_title(f"{img_path.name} — {n_boxes} box(es)", fontsize=9)
        ax.axis("off")
    for ax in np.array(axes).ravel()[len(picks):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()

show_kitti_samples(KITTI_DIR)

### 4.5 Convert KITTI → TFRecords

`dataset_convert` shards the dataset and carves out the validation fold. `val_split` here is a
**percentage**, and `partition_mode: "random"` with `num_partitions: 2` is what produces the
train/val division — so no manual file splitting is needed.

In [ ]:
from string import Template

TFRECORDS_SPEC = Template('''kitti_config {
  root_directory_path: "$KITTI_ROOT"
  image_dir_name: "images"
  label_dir_name: "labels"
  image_extension: ".png"
  partition_mode: "random"
  num_partitions: 2
  val_split: $VAL_PCT
  num_shards: 10
}
image_directory_path: "$KITTI_ROOT"
''')

tfrecords_spec_path = SPECS_DIR / "detectnet_v2_tfrecords_kitti.txt"
tfrecords_spec_path.write_text(TFRECORDS_SPEC.substitute(
    KITTI_ROOT=to_container(KITTI_DIR),
    VAL_PCT=int(VAL_SPLIT * 100),
))
print(tfrecords_spec_path.read_text())

In [ ]:
# Run the conversion inside the container.
tao(
    "detectnet_v2 dataset_convert "
    f"-d {to_container(tfrecords_spec_path)} "
    f"-o {to_container(TFRECORDS_DIR)}/kitti_trainval"
)

In [ ]:
shards = sorted(TFRECORDS_DIR.glob("*"))
print(f"{len(shards)} shard(s) written to {TFRECORDS_DIR}")
for s in shards[:6]:
    print(f"  {s.name}  ({s.stat().st_size / 1e6:.1f} MB)")
if len(shards) > 6:
    print(f"  ... and {len(shards) - 6} more")
assert shards, "dataset_convert produced no shards — check the spec paths above"

---
## 5. Specify training parameters

The spec file is the whole experiment: dataset wiring, augmentation, backbone, loss weights, and
the optimizer schedule. The knobs worth touching first:

| Knob | Where | Effect |
|---|---|---|
| `batch_size_per_gpu` | `training_config` | Raise until GPU memory is full; halve if you OOM |
| `num_epochs` | `training_config` | Main wall-clock lever (~1 h at 80 epochs on an A6000) |
| `max_learning_rate` | `learning_rate` | Scale roughly with batch size |
| `minimum_detection_ground_truth_overlap` | `evaluation_config` | IoU threshold for a true positive |
| `dbscan_confidence_threshold` | `postprocessing_config` | Precision/recall trade-off at inference |

Synthetic data has no sensor noise, so the **color augmentation** block matters more than usual —
it is most of what closes the sim-to-real gap.

In [ ]:
TRAIN_SPEC = Template('''random_seed: $SEED
dataset_config {
  data_sources {
    tfrecords_path: "$TFRECORDS/*"
    image_directory_path: "$KITTI_ROOT"
  }
  image_extension: "png"
  target_class_mapping {
    key: "$CLS"
    value: "$CLS"
  }
  validation_fold: 0
}
augmentation_config {
  preprocessing {
    output_image_width: $W
    output_image_height: $H
    output_image_channel: 3
    min_bbox_width: 1.0
    min_bbox_height: 1.0
  }
  spatial_augmentation {
    hflip_probability: 0.5
    zoom_min: 1.0
    zoom_max: 1.0
    translate_max_x: 8.0
    translate_max_y: 8.0
  }
  color_augmentation {
    hue_rotation_max: 25.0
    saturation_shift_max: 0.20
    contrast_scale_max: 0.10
    contrast_center: 0.5
  }
}
model_config {
  pretrained_model_file: "$PRETRAINED"
  num_layers: $LAYERS
  arch: "resnet"
  use_batch_norm: true
  all_projections: true
  objective_set {
    bbox {
      scale: 35.0
      offset: 0.5
    }
    cov {
    }
  }
}
postprocessing_config {
  target_class_config {
    key: "$CLS"
    value {
      clustering_config {
        clustering_algorithm: DBSCAN
        dbscan_confidence_threshold: 0.9
        dbscan_eps: 0.25
        dbscan_min_samples: 1
        minimum_bounding_box_height: 20
      }
    }
  }
}
evaluation_config {
  validation_period_during_training: 10
  first_validation_epoch: 10
  average_precision_mode: INTEGRATE
  minimum_detection_ground_truth_overlap {
    key: "$CLS"
    value: 0.5
  }
  evaluation_box_config {
    key: "$CLS"
    value {
      minimum_height: 20
      maximum_height: 9999
      minimum_width: 10
      maximum_width: 9999
    }
  }
}
cost_function_config {
  target_classes {
    name: "$CLS"
    class_weight: 1.0
    coverage_foreground_weight: 0.05
    objectives {
      name: "cov"
      initial_weight: 1.0
      weight_target: 1.0
    }
    objectives {
      name: "bbox"
      initial_weight: 10.0
      weight_target: 10.0
    }
  }
  enable_autoweighting: true
  max_objective_weight: 0.9999
  min_objective_weight: 0.0001
}
training_config {
  batch_size_per_gpu: $BATCH
  num_epochs: $EPOCHS
  checkpoint_interval: 10
  learning_rate {
    soft_start_annealing_schedule {
      min_learning_rate: $MIN_LR
      max_learning_rate: $MAX_LR
      soft_start: 0.1
      annealing: 0.7
    }
  }
  regularizer {
    type: L1
    weight: 3e-9
  }
  optimizer {
    adam {
      epsilon: 1e-08
      beta1: 0.9
      beta2: 0.999
    }
  }
  cost_scaling {
    enabled: false
    initial_exponent: 20.0
    increment: 0.005
    decrement: 1.0
  }
}
bbox_rasterizer_config {
  target_class_config {
    key: "$CLS"
    value {
      cov_center_x: 0.5
      cov_center_y: 0.5
      cov_radius_x: 0.4
      cov_radius_y: 0.4
      bbox_min_radius: 1.0
    }
  }
  deadzone_radius: 0.67
}
''')

train_spec_path = SPECS_DIR / "detectnet_v2_train.txt"
train_spec_path.write_text(TRAIN_SPEC.substitute(
    SEED=RANDOM_SEED,
    TFRECORDS=f"{to_container(TFRECORDS_DIR)}/kitti_trainval",
    KITTI_ROOT=to_container(KITTI_DIR),
    CLS=TARGET_CLASS,
    W=IMAGE_WIDTH,
    H=IMAGE_HEIGHT,
    PRETRAINED=to_container(PRETRAINED_WEIGHTS),
    LAYERS=ARCH_LAYERS,
    BATCH=BATCH_SIZE,
    EPOCHS=NUM_EPOCHS,
    MIN_LR=f"{LEARNING_RATE / 100:.2e}",
    MAX_LR=f"{LEARNING_RATE:.2e}",
))
print(f"wrote {train_spec_path}\n")
print(train_spec_path.read_text())

---
## 6. Train the model

Roughly **one hour for 80 epochs on an RTX A6000**. Checkpoints land in
`results/detectnet_v2/weights/` every 10 epochs, so an interrupted run is resumable — rerun the
same cell and TAO picks up from the newest checkpoint.

Watch the validation block that prints every 10 epochs: **mean average precision** should climb
and then plateau. If it is still rising at the last epoch, raise `NUM_EPOCHS`.

In [ ]:
TRAIN_OUT = RESULTS_DIR / "detectnet_v2"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)

tao(
    "detectnet_v2 train "
    f"-e {to_container(train_spec_path)} "
    f"-r {to_container(TRAIN_OUT)} "
    f"-k {KEY} "
    f"--gpus {NUM_GPUS}"
)

In [ ]:
# Locate the final checkpoint.
ckpts = sorted((TRAIN_OUT / "weights").glob("*.hdf5"))
assert ckpts, f"no checkpoints under {TRAIN_OUT / 'weights'} — did training finish?"
TRAINED_MODEL = ckpts[-1]
print(f"{len(ckpts)} checkpoint(s); using {TRAINED_MODEL.name}")
for c in ckpts:
    print(f"  {c.name}  ({c.stat().st_size / 1e6:.0f} MB)")

---
## 7. Evaluate on test data

`evaluate` runs the model over the validation fold and reports **average precision per class** at
the IoU threshold from `evaluation_config`.

Rough read on synthetic-only training for a single class: **AP > 0.85** on synthetic validation is
expected and easy; the number that actually matters is AP on *real* images, which is typically
much lower. That gap is the sim-to-real gap, and it is what more aggressive domain randomization
in Replicator is for.

In [ ]:
import re

eval_log = RESULTS_DIR / "evaluate.log"

proc = subprocess.run(
    f"docker run --rm --gpus all --shm-size=16g --ulimit memlock=-1 --ulimit stack=67108864 "
    f"-u {os.getuid()}:{os.getgid()} -v {PROJECT_DIR}:{TAO_DIR} -w {TAO_DIR} {TAO_IMAGE} "
    f"detectnet_v2 evaluate -e {to_container(train_spec_path)} "
    f"-m {to_container(TRAINED_MODEL)} -k {KEY}",
    shell=True, capture_output=True, text=True,
)
output = proc.stdout + proc.stderr
eval_log.write_text(output)
print(output[-4000:])          # tail; full log saved to results/evaluate.log

In [ ]:
# Parse the AP table out of the evaluation log.
def parse_ap(log_text):
    """Extract {class: average_precision} from detectnet_v2 evaluate output."""
    aps, mean_ap = {}, None
    for line in log_text.splitlines():
        m = re.match(r"^\s*(\w[\w\-]*)\s+([01]\.\d+)\s*$", line)
        if m and m.group(1).lower() not in {"class", "name"}:
            aps[m.group(1)] = float(m.group(2))
        m2 = re.search(r"[Mm]ean average precision[:\s]+([01]\.\d+)", line)
        if m2:
            mean_ap = float(m2.group(1))
    if mean_ap is None and aps:
        mean_ap = sum(aps.values()) / len(aps)
    return aps, mean_ap

aps, mean_ap = parse_ap(output)
if aps:
    print("Average precision @ IoU 0.5")
    print("-" * 34)
    for cls, ap in sorted(aps.items()):
        bar = "#" * int(ap * 24)
        print(f"  {cls:<16} {ap:0.4f}  {bar}")
    print("-" * 34)
    print(f"  {'mAP':<16} {mean_ap:0.4f}")
else:
    print("Could not parse an AP table — inspect results/evaluate.log directly.")

---
## 8. Visualize results

Numbers tell you *how much* the model is wrong; overlays tell you *how*. Run inference on a
handful of images and look for the characteristic failure modes: duplicate boxes on one object
(lower `dbscan_eps`), boxes that hug only part of the object (train longer), or confident
detections on background clutter (more distractors in the Replicator scene).

In [ ]:
INFER_SPEC = Template('''inferencer_config {
  target_classes: "$CLS"
  image_width: $W
  image_height: $H
  image_channels: 3
  batch_size: 16
  gpu_index: 0
  tlt_config {
    model: "$MODEL"
  }
}
bbox_handler_config {
  kitti_dump: true
  disable_overlay: false
  overlay_linewidth: 2
  classwise_bbox_handler_config {
    key: "$CLS"
    value {
      confidence_model: "aggregate_cov"
      output_map: "$CLS"
      bbox_color {
        R: 0
        G: 255
        B: 0
      }
      clustering_config {
        clustering_algorithm: DBSCAN
        dbscan_confidence_threshold: 0.9
        dbscan_eps: 0.25
        dbscan_min_samples: 1
        minimum_bounding_box_height: 20
      }
    }
  }
}
''')

infer_spec_path = SPECS_DIR / "detectnet_v2_inference.txt"
infer_spec_path.write_text(INFER_SPEC.substitute(
    CLS=TARGET_CLASS, W=IMAGE_WIDTH, H=IMAGE_HEIGHT,
    MODEL=to_container(TRAINED_MODEL),
))
print(infer_spec_path.read_text())

In [ ]:
# Run inference on a small sample so the overlay step is quick.
INFER_IN  = RESULTS_DIR / "infer_input"
INFER_OUT = RESULTS_DIR / "infer_output"
INFER_IN.mkdir(exist_ok=True)

sample = random.Random(RANDOM_SEED + 1).sample(images, min(12, len(images)))
for p in sample:
    shutil.copy(p, INFER_IN / p.name)
print(f"{len(sample)} images staged in {INFER_IN}")

tao(
    "detectnet_v2 inference "
    f"-e {to_container(infer_spec_path)} "
    f"-i {to_container(INFER_IN)} "
    f"-o {to_container(INFER_OUT)} "
    f"-k {KEY}"
)

In [ ]:
# TAO writes rendered overlays to images_annotated/ and KITTI-format detections to labels/.
annotated = sorted((INFER_OUT / "images_annotated").glob("*"))
print(f"{len(annotated)} annotated image(s) in {INFER_OUT / 'images_annotated'}")

if annotated:
    picks = annotated[:6]
    cols = 2
    rows = (len(picks) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(9 * cols, 5 * rows))
    for ax, p in zip(np.array(axes).ravel(), picks):
        ax.imshow(Image.open(p))
        ax.set_title(p.name, fontsize=9)
        ax.axis("off")
    for ax in np.array(axes).ravel()[len(picks):]:
        ax.axis("off")
    fig.suptitle("Model predictions", fontsize=14)
    fig.tight_layout()
    plt.show()

In [ ]:
# Side-by-side: ground truth (green) vs prediction (magenta), with confidence.
def compare_gt_vs_pred(stem):
    img_path = KITTI_DIR / "images" / f"{stem}.png"
    gt_path  = KITTI_DIR / "labels" / f"{stem}.txt"
    pr_path  = INFER_OUT / "labels" / f"{stem}.txt"
    if not img_path.exists():
        print(f"no such image: {stem}")
        return

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.imshow(Image.open(img_path))

    def draw(path, color, label, show_conf=False):
        n = 0
        if not path.exists():
            return n
        for line in path.read_text().splitlines():
            p = line.split()
            if len(p) < 8:
                continue
            x1, y1, x2, y2 = (float(v) for v in p[4:8])
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor="none",
                label=label if n == 0 else None,
            ))
            if show_conf and len(p) >= 16:
                ax.text(x1, y1 - 4, f"{float(p[15]):.2f}", color=color, fontsize=9,
                        bbox=dict(facecolor="black", alpha=0.5, pad=1))
            n += 1
        return n

    n_gt = draw(gt_path, "lime", "ground truth")
    n_pr = draw(pr_path, "magenta", "prediction", show_conf=True)
    ax.set_title(f"{stem} — {n_gt} GT / {n_pr} predicted", fontsize=11)
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

for p in sample[:3]:
    compare_gt_vs_pred(p.stem)

---
## 9. Optional — prune and export for deployment

Pruning drops low-magnitude filters, typically **shrinking the model 5-10×** with a small accuracy
cost that retraining recovers. Export produces a deployable `.onnx`/`.etlt` for TensorRT and
DeepStream.

Only worth doing once you are happy with the metrics from Step 7.

In [ ]:
RUN_EXPORT = False   # flip to True when you are ready to deploy

if RUN_EXPORT:
    pruned = RESULTS_DIR / "detectnet_v2_pruned" / "resnet18_pruned.hdf5"
    pruned.parent.mkdir(parents=True, exist_ok=True)
    tao(
        "detectnet_v2 prune "
        f"-m {to_container(TRAINED_MODEL)} "
        f"-o {to_container(pruned)} "
        f"-eq union -pth 0.8 -k {KEY}"
    )
    # NOTE: retrain on `pruned` with the same spec (set pretrained_model_file to it and
    # load_graph: true in model_config) before exporting, to recover the lost accuracy.

    exported = RESULTS_DIR / "detectnet_v2_export" / "resnet18_detector.onnx"
    exported.parent.mkdir(parents=True, exist_ok=True)
    tao(
        "detectnet_v2 export "
        f"-m {to_container(TRAINED_MODEL)} "
        f"-o {to_container(exported)} "
        f"-k {KEY} --data_type fp16"
    )
    print(f"exported: {exported}")
else:
    print("export skipped — set RUN_EXPORT = True to enable")

---
## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `docker: no matching manifest for linux/arm64` | The TF1 TAO container is amd64-only. Run Steps 2-7 on an x86_64 host (see Step 0). |
| `Resource exhausted: OOM` during training | Halve `BATCH_SIZE` in Step 1 and re-run Steps 5-6. |
| `dataset_convert` writes 0 shards | Paths in the spec are *container* paths. Confirm `to_container()` output matches the mount in `~/.tao_mounts.json`. |
| mAP stuck at 0.0 | `target_class_mapping.key` must match column 0 of the KITTI labels **exactly and in lowercase**. Re-check Step 4.3's class histogram. |
| Loss goes to `NaN` | Learning rate too high for the batch size — drop `LEARNING_RATE` by 10×, or enable `cost_scaling`. |
| Many duplicate boxes per object | Lower `dbscan_eps` (e.g. 0.15) in the postprocessing config. |
| Good synthetic AP, poor real-world AP | Sim-to-real gap. Add distractors, lighting, texture and camera-pose randomization in Replicator; increase `color_augmentation`. |
| `unauthorized: authentication required` on pull | `NGC_API_KEY` missing or expired — regenerate and re-run Step 2.2. |

## Next steps

- Re-generate synthetic data with wider domain randomization and compare AP — this is the highest-leverage loop in the whole workflow.
- Mix in a small set of labeled **real** images (even 5-10%) and re-train; this usually closes most of the remaining gap.
- Deploy the exported model with DeepStream or a TensorRT runtime.